# 03a - Mesin Vektor Pendukung (SVM)
Materi Modul 3: pengantar SVM, klasifikasi dengan SVM, fungsi biaya SVM (hinge loss), regularisasi dalam SVM.

SVM mencari **hyperplane** yang memisahkan kelas dengan **margin terbesar**. Titik yang menentukan margin disebut **support vector**.

Fungsi biaya (hinge loss): $\max(0, 1 - y \cdot f(x))$ + penalti regularisasi. Parameter `C` mengatur trade-off margin lebar vs kesalahan klasifikasi.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC, LinearSVC
from sklearn.linear_model import SGDClassifier

In [ ]:
def plot_batas(model, X, y, judul=''):
    # menggambar batas keputusan untuk data 2 fitur
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
    zz = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    plt.contourf(xx, yy, zz, alpha=0.3, cmap='coolwarm')
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', edgecolor='k', s=25)
    plt.title(judul)

## 1. Hinge loss

In [ ]:
margin = np.linspace(-2, 3, 200)           # y * f(x)
hinge = np.maximum(0, 1 - margin)
logistik = np.log(1 + np.exp(-margin))
nol_satu = (margin < 0).astype(float)

plt.plot(margin, hinge, label='Hinge (SVM)')
plt.plot(margin, logistik, label='Log loss (regresi logistik)')
plt.plot(margin, nol_satu, '--', label='0-1 loss')
plt.xlabel('y * f(x)'); plt.ylabel('Loss'); plt.ylim(0, 3); plt.legend()
plt.title('Perbandingan fungsi biaya'); plt.show()

## 2. SVM linier pada 2 fitur + support vector

In [ ]:
wine = pd.read_csv('../datasets/wine.csv')
duadua = wine[wine['target'].isin([0, 1])]
X2 = StandardScaler().fit_transform(duadua[['alcohol', 'flavanoids']])
y2 = duadua['target'].values

svm = SVC(kernel='linear', C=1).fit(X2, y2)
plot_batas(svm, X2, y2, 'SVM linier')
plt.scatter(svm.support_vectors_[:, 0], svm.support_vectors_[:, 1], s=120, facecolors='none', edgecolors='k', linewidths=1.5)
plt.show()
print('Jumlah support vector:', svm.n_support_, '(per kelas)')

## 3. Pengaruh `C` (regularisasi)
**C kecil** = margin lebar, toleran kesalahan (regularisasi kuat). **C besar** = margin sempit, berusaha benar semua (risiko overfit).

In [ ]:
fig = plt.figure(figsize=(15, 4))
for i, c in enumerate([0.01, 1, 100], start=1):
    plt.subplot(1, 3, i)
    m = SVC(kernel='linear', C=c).fit(X2, y2)
    plot_batas(m, X2, y2, f'C = {c} | support vector = {m.support_vectors_.shape[0]}')
plt.tight_layout(); plt.show()

## 4. SVM pada data lengkap (Breast Cancer)

In [ ]:
df = pd.read_csv('../datasets/breast_cancer.csv')
X, y = df.drop(columns='target'), df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print('C      | akurasi train | akurasi test | support vector')
for c in [0.001, 0.01, 0.1, 1, 10, 100]:
    m = make_pipeline(StandardScaler(), SVC(kernel='linear', C=c)).fit(X_train, y_train)
    print(f'{c:<6} | {m.score(X_train, y_train):.4f}        | {m.score(X_test, y_test):.4f}       | {m[-1].support_vectors_.shape[0]}')

## 5. `LinearSVC` dan `SGDClassifier(loss='hinge')`
Keduanya SVM linier yang lebih cepat untuk data besar.

In [ ]:
linsvc = make_pipeline(StandardScaler(), LinearSVC(C=1, max_iter=10000)).fit(X_train, y_train)
sgd = make_pipeline(StandardScaler(), SGDClassifier(loss='hinge', random_state=42)).fit(X_train, y_train)

print('SVC linier      :', round(make_pipeline(StandardScaler(), SVC(kernel='linear')).fit(X_train, y_train).score(X_test, y_test), 4))
print('LinearSVC       :', round(linsvc.score(X_test, y_test), 4))
print('SGD (hinge)     :', round(sgd.score(X_test, y_test), 4))

## Catatan
- (apa itu margin dan support vector, arti C, beda hinge loss dan log loss, dll.)